## nb_03_3_game_goalie_stats_silver

Cleans and validates bronze table and writes the result to a silver table
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [16]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, 6821e94b-b506-42a3-ae9a-54e2886cc611, 30, Finished, Available, Finished, False)

### Load common db functions

In [17]:
%run nb_00_dbutils

StatementMeta(, 6821e94b-b506-42a3-ae9a-54e2886cc611, 43, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded and the pipeline against the real `bronze.tablenmae` / `silver.tablename`
tables is skipped:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [18]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 6821e94b-b506-42a3-ae9a-54e2886cc611, 44, Finished, Available, Finished, False)

### Config

In [19]:
BRONZE_TABLE = "bronze.game_goalie_stats"
SILVER_TABLE = "silver.game_goalie_stats"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "game_id",
    "player_id",
    "team_id",
    "timeOnIce",
    "shots",
    "saves",
    "decision"
]

no_null_cols: list[str] = [
    "game_id",
    "player_id",
    "team_id",
    "timeOnIce",
    "shots",
    "saves",
]


# Natural key used to de-duplicate rows
DEDUPE_KEYS: list[str] = ["game_id", "player_id", "team_id"]
PRIMARY_KEYS: list[str] = ["game_id", "player_id", "team_id"]

# Prevents table from loading when called from tests
# Check integrity against silver tables as they clean
if RUN_PIPELINE:

    # Set up foreign keys for integrity checks
    GAME_FOREIGN_KEYS: list[str] = ["game_id"]
    game: DataFrame = load_table(spark, "silver.game", GAME_FOREIGN_KEYS)

    PLAYER_INFO_FOREIGN_KEYS: list[str] = ["player_id"]
    player_info: DataFrame = load_table(spark, "silver.player_info", PLAYER_INFO_FOREIGN_KEYS)

    TEAM_INFO_FOREIGN_KEYS: list[str] = ["team_id"]
    team_info: DataFrame = load_table(spark, "silver.team_info", TEAM_INFO_FOREIGN_KEYS)


StatementMeta(, 6821e94b-b506-42a3-ae9a-54e2886cc611, 45, Finished, Available, Finished, False)

✅ Loaded silver.game: 23730 rows
✅ Loaded silver.player_info: 3925 rows
✅ Loaded silver.team_info: 33 rows


## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [21]:
if RUN_PIPELINE:
    df = load_table(spark, BRONZE_TABLE, columns=required_cols)

    df = remove_duplicates(df, columns=DEDUPE_KEYS)
    df = validate_no_nulls(df, columns=no_null_cols)

    df = drop_foreign_key_violations(df, ftable=game, keys=GAME_FOREIGN_KEYS)
    df = drop_foreign_key_violations(df, ftable=team_info, keys=TEAM_INFO_FOREIGN_KEYS)
    df = drop_foreign_key_violations(df, ftable=player_info, keys=PLAYER_INFO_FOREIGN_KEYS)

    df = validate_foreign_keys(df, ftable=game, keys=GAME_FOREIGN_KEYS)
    df = validate_foreign_keys(df, ftable=team_info, keys=TEAM_INFO_FOREIGN_KEYS)
    df = validate_foreign_keys(df, ftable=player_info, keys=PLAYER_INFO_FOREIGN_KEYS)

    # Validate primary key(s) to ensure integrity
    df = validate_primary_keys(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 6821e94b-b506-42a3-ae9a-54e2886cc611, 47, Finished, Available, Finished, False)

✅ Loaded bronze.game_goalie_stats: 56656 rows
🔁 Removed 5493 duplicate row(s) based on ['game_id', 'player_id', 'team_id']
⚠️ Dropping 20 rows with foreign key violations for ['game_id']
✅ Foreign key check passed — ['team_id']
✅ Foreign key check passed — ['player_id']
✅ Foreign key check passed — ['game_id']
✅ Foreign key check passed — ['team_id']
✅ Foreign key check passed — ['player_id']
✅ Primary key check passed — ['game_id', 'player_id', 'team_id'] is unique across 51143 row(s)
✅ Wrote silver.game_goalie_stats (51143 rows, 7 columns)
🏁 Silver load complete.
